##Transforming results data

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/3.silver_helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

## Schema Shaping

In [0]:
results_df = (
  spark.table(bronze_table)
       .filter(F.col("batch_id") == v_batch_id)
       .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingestion_timestamp",
                "source_file",
                "batch_id")
       .withColumnsRenamed({
                "constructorId": "constructor_id",
                "driverId": "driver_id",
                "raceName": "race_name",
                "date": "race_date",
                "grid": "grid_position",
                "laps": "completed_laps",
                "number": "car_number",
                "position": "final_position",
                "positionText": "final_position_text"
        })
       
)


## Data Quality

In [0]:
results_valid_df = (results_df
                    .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull() 
        )
       .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
       .withColumn('race_name', F.initcap(F.col("race_name")))
)

In [0]:
display(results_df.count() - results_valid_df.count())

## Value level transformations

In [0]:
results_final_df = (
    results_valid_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
)


## Writing to silver delta table

In [0]:
results_columns_to_update = [
    "season",
    "round",
    "constructor_id",
    "driver_id",
    "race_date",
    "race_name",
    "grid_position",
    "completed_laps",
    "car_number",
    "points",
    "final_position",
    "final_position_text",
    "status",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
]

write_to_silver(
    results_final_df, 
    silver_table,
    merge_condition = "t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update = results_columns_to_update
    )

In [0]:
spark.table(silver_table).display()